In [9]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from monai.data import Dataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, 
    ScaleIntensityRangePercentilesd, Resized, ToTensord
)
from monai.networks.nets import EfficientNetBN

In [ ]:

import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from monai.data import Dataset
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRangePercentilesd,
    Resized,
    ToTensord
)
from monai.networks.nets import EfficientNetBN

# =========================================================
# 1. AUTOMATIC FILE SEARCH IN THE 'SAMPLES' FOLDER
# =========================================================
# Define the directory path (adjust if using absolute paths on Windows)
SAMPLES_DIR = "../data/samples"

# RECURSIVE Search: Look for .dcm or .DCM files in the folder and ALL subfolders
dcm_files = glob.glob(os.path.join(SAMPLES_DIR, "**", "*.dcm"), recursive=True) + \
            glob.glob(os.path.join(SAMPLES_DIR, "**", "*.DCM"), recursive=True)

# Safety check to prevent IndexError
if len(dcm_files) == 0:
    raise FileNotFoundError(
        f"No .dcm files found in path '{SAMPLES_DIR}'. "
        f"Verify the folder name. Detected content: {os.listdir(SAMPLES_DIR) if os.path.exists(SAMPLES_DIR) else 'Folder does not exist'}"
    )

print(f"✅ Success! Found {len(dcm_files)} DICOM files.")

# Build the data structure required by MONAI
data_list = []
for file_path in dcm_files:
    data_list.append({
        "image": file_path,
        "label": 0  # Default label (0 = Normal/Healthy). You can link this to your CSV later.
    })

print("Sample first element configured:", data_list[0])


# =========================================================
# 2. MONAI TRANSFORMS (Medical Preprocessing)
# =========================================================
transforms = Compose([
    LoadImaged(keys=["image"]),                       # Load the .dcm file from disk
    EnsureChannelFirstd(keys=["image"]),              # Set channel-first format (1, Height, Width)
    ScaleIntensityRangePercentilesd(                  # Radiological intensity normalization
        keys=["image"], lower=1, upper=99, b_min=0.0, b_max=1.0
    ),
    Resized(keys=["image"], spatial_size=(380, 380)), # Native input resolution for EfficientNet-B4
    ToTensord(keys=["image"])                          # Convert to PyTorch Tensor
])


# =========================================================
# 3. DATASET AND DATALOADER CREATION
# =========================================================
monai_dataset = Dataset(data=data_list, transform=transforms)

# DataLoader to send images in batches to the neural network
train_loader = DataLoader(
    monai_dataset, 
    batch_size=2,      # Adjust batch size based on your RAM/GPU memory
    shuffle=True, 
    num_workers=0      # '0' recommended for Jupyter/Google Colab to avoid threading issues
)


# =========================================================
# 4. INSTANTIATE EFFICIENTNET MODEL WITHIN MONAI
# =========================================================
# EfficientNet-B4 configured for 1 channel (native DICOM grayscale)
model = EfficientNetBN(
    model_name="efficientnet-b4",
    spatial_dims=2,     # 2D for X-rays or CT slices
    in_channels=1,      # 1 input channel (Grayscale)
    num_classes=2       # Output: 2 classes (e.g., 0 = No Lesion, 1 = Lesion)
)

# Set up GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"🖥️ Model loaded on: {device}")


# =========================================================
# 5. TEST INFERENCE ON A SINGLE BATCH
# =========================================================
model.eval()
with torch.no_grad():
    for batch in train_loader:
        images = batch["image"].to(device)  # Input tensor shape: [Batch, 1, 380, 380]
        labels = batch["label"].to(device)
        
        # Inference
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)
        
        print("\n--- TEST RESULTS ---")
        print("Input batch shape:", images.shape)
        print("Network outputs (Logits):", outputs)
        print("Final predictions:", predictions.cpu().numpy())
        break


✅ Success! Found 10 DICOM files.
Sample first element configured: {'image': '../data/samples/1.2.826.0.1.3680043.8.498.10200533543072707116236088903375502248.dcm', 'label': 0}
🖥️ Model loaded on: cuda

--- TEST RESULTS ---
Input batch shape: torch.Size([2, 1, 380, 380])
Network outputs (Logits): metatensor([[4.0941, 2.4268],
        [3.9772, 2.1149]], device='cuda:0')
Final predictions: [0 0]
